# YOLOv8 Evaluation — AI Racing Tech (single-class `car`)

Evaluates a fine-tuned YOLOv8-seg checkpoint against a held-out set of SAM2-labeled
rosbags. Supports both prime-lens and fisheye-lens camera views and expects
single-class (`car`) labels.

**Inputs**
- `DATASETS_DIR`: directory containing held-out bags in the format
  `bag/images/*.jpg` and `bag/labels/*.txt` (YOLO segmentation labels,
  class id `0`). If you only have SAM2 masks, run `sam2_finetune_med.py --format_only`
  first to generate labels.
- `DATA_YAML`: path to `testing_data.yaml` (single class, `0: car`).
- `MODEL_PATH`: path to the fine-tuned `.pt` or `.onnx` weights.

**Output**
A flat `eval_data/eval/{images,labels}/` directory built from all bags, followed
by `model.val()` metrics (box + mask mAP).


In [ ]:
import os
import random
import shutil

import torch
import tqdm
import ultralytics
from ultralytics import YOLO

## Config — edit these paths for your machine

In [ ]:
# Absolute or relative paths. Defaults assume the layout documented in the README:
#   <workspace>/YOLOv8-Fine-Tune/   <- this repo
#   <workspace>/holdout_bags/       <- held-out SAM2-labeled bags
#   <workspace>/eval_data/          <- generated by this notebook
#   <workspace>/models/best.pt      <- fine-tuned checkpoint
CURR_DIR = os.getcwd()                       # .../YOLOv8-Fine-Tune/src
WORKSPACE_DIR = os.path.dirname(CURR_DIR)    # .../YOLOv8-Fine-Tune
PARENT_DIR = os.path.dirname(WORKSPACE_DIR)  # .../

DATASETS_DIR = os.path.join(PARENT_DIR, 'holdout_bags')
EVAL_OUT_DIR = os.path.join(PARENT_DIR, 'eval_data')
DATA_YAML    = os.path.join(WORKSPACE_DIR, 'testing_data.yaml')
MODEL_PATH   = os.path.join(WORKSPACE_DIR, 'models', 'best.pt')

IMG_SIZE = 1056  # must match training imgsz

# Keep empty (no-car) frames so precision is measured honestly.
KEEP_EMPTY_FRAMES = True
PERCENTAGE_EMPTY_FRAMES_TO_KEEP = 1.0

# Per-dataset frame weighting (name substring -> keep probability or duplication factor).
DATASET_WEIGHTS = {}

for p in [DATASETS_DIR, DATA_YAML, MODEL_PATH]:
    print(p, '->', 'OK' if os.path.exists(p) else 'MISSING')

## Build flat eval dataset from held-out bags

In [ ]:
def _generate_empty_label(path):
    with open(path, 'w') as f:
        f.write('')

def _choose_weight(img_src, weighted, removed):
    dataset_name = img_src.split(os.sep)[-3]
    for key, w in DATASET_WEIGHTS.items():
        if key in dataset_name:
            if w > 1:
                weighted[key] = weighted.get(key, 0) + (w - 1)
                return int(w)
            if random.random() < w:
                return 1
            removed[key] = removed.get(key, 0) + 1
            return 0
    return 1

def _copy_pair(label_src, img_src, label_dst, img_dst, counters):
    weighted, removed = counters['weighted'], counters['removed']
    if not os.path.exists(label_src):
        if not KEEP_EMPTY_FRAMES or random.random() >= PERCENTAGE_EMPTY_FRAMES_TO_KEEP:
            return
        counters['empty'] += 1
        for i in range(_choose_weight(img_src, weighted, removed)):
            shutil.copy(img_src, img_dst[:-4] + f'_{i}.jpg')
            _generate_empty_label(label_dst[:-4] + f'_{i}.txt')
        return
    # Normalize any legacy class ids to 0 (single-class car) without mutating the source.
    with open(label_src) as f:
        lines = f.readlines()
    fixed = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        parts[0] = '0'
        fixed.append(' '.join(parts) + '\n')
    for i in range(_choose_weight(img_src, weighted, removed)):
        shutil.copy(img_src, img_dst[:-4] + f'_{i}.jpg')
        with open(label_dst[:-4] + f'_{i}.txt', 'w') as f:
            f.writelines(fixed)

def format_eval_dataset(datasets_dir, out_dir, data_yaml):
    assert os.path.exists(datasets_dir), f'Missing {datasets_dir}'
    assert os.path.exists(data_yaml),    f'Missing {data_yaml}'

    pairs = []
    for bag in sorted(os.listdir(datasets_dir)):
        bag_imgs = os.path.join(datasets_dir, bag, 'images')
        bag_lbls = os.path.join(datasets_dir, bag, 'labels')
        if not os.path.isdir(bag_imgs):
            continue
        for img in sorted(os.listdir(bag_imgs)):
            label = os.path.join(bag_lbls, os.path.splitext(img)[0] + '.txt')
            pairs.append((os.path.join(bag_imgs, img), label, bag))

    random.seed(0)
    random.shuffle(pairs)

    if os.path.exists(out_dir):
        print(f'Deleting existing {out_dir}/')
        shutil.rmtree(out_dir)
    os.makedirs(os.path.join(out_dir, 'eval', 'images'))
    os.makedirs(os.path.join(out_dir, 'eval', 'labels'))

    counters = {'empty': 0, 'weighted': {}, 'removed': {}}
    kept = 0
    for uid, (img_src, label_src, bag) in enumerate(tqdm.tqdm(pairs, desc='Copying')):
        stem = os.path.splitext(os.path.basename(img_src))[0]
        img_dst   = os.path.join(out_dir, 'eval', 'images', f'{bag}_{stem}_{uid}.jpg')
        label_dst = os.path.join(out_dir, 'eval', 'labels', f'{bag}_{stem}_{uid}.txt')
        _copy_pair(label_src, img_src, label_dst, img_dst, counters)
        kept += 1

    shutil.copy(data_yaml, out_dir)
    print(f'Eval frames:        {kept}')
    print(f'Empty frames kept:  {counters["empty"]}')
    print(f'Weighted additions: {counters["weighted"]}')
    print(f'Removed frames:     {counters["removed"]}')

format_eval_dataset(DATASETS_DIR, EVAL_OUT_DIR, DATA_YAML)

## Run validation

In [ ]:
print('CUDA available:', torch.cuda.is_available())
print('Torch CUDA:    ', torch.version.cuda)

model = YOLO(MODEL_PATH, task='segment')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

results = model.val(data=DATA_YAML, imgsz=IMG_SIZE)
print(results.results_dict)

## Notes
- `MODEL_PATH` can point to either a `.pt` or an `.onnx` file. For ONNX,
  `imgsz` must match the export size (`sam2_finetune_med.py` exports at
  `IMG_SIZE = 1056`).
- `testing_data.yaml` must have `nc: 1` and `names: {0: car}` — the label
  normalization step above rewrites any legacy class ids to `0`.
- Results are written to `runs/segment/val*/` under the current working dir.
